In [44]:
# Libraries
import numpy as np
import pandas as pd

## 1. Simplified EDA

First of all, we are going to load the dataset and perform exploratory data analysis, in order to get a good idea about what we are working with.

In [45]:
# First, we load the dataset, it is a PKL file.
df = pd.read_pickle("ecommerce_4.pkl")

First, we are going to see the dimensions, the types of variables present and the missing values they have, and whether there are constant columns

In [46]:
rows, cols = df.shape
print(f"The dataset has {rows} rows and {cols} variables\n")

# Let's list the variables we have
variables = pd.DataFrame({
    "Variable": df.columns,
    "Type": df.dtypes.values,
    "Null Values": df.isnull().sum().values,
    "Constant Column": (df.nunique() == 1).map({True: 'Yes', False: 'No'}).values
})

# Show the table
print(variables.to_string(index=False))

The dataset has 20240 rows and 26 variables

                Variable    Type  Null Values Constant Column
             Customer_ID  object            0              No
                     Age float64          974              No
                  Gender  object          612              No
          Credit_Balance float64         2245              No
          Signup_Quarter  object            0              No
        Membership_Years float64            0              No
             Logged_Freq float64            0              No
              Logged_Avg float64         1581              No
       Pages_Per_Session float64         1223              No
          Cart_Drop_Rate float64            0              No
          Wishlist_Items float64         1636              No
         Total_Purchases float64            0              No
     Average_Order_Value float64            0              No
   Days_without_Purchase float64         1222              No
     Discount_Usage_Rate 

Our dataset has 20240 instances and 26 features. The target variable, *Canceled*, appears to be numerical and discrete, but in fact we will see it is categorical, taking 0 or 1 as values. For the other variables we get a mix of numerical continuous, discrete and categorical. Talking about null values, we see that there are several features with some of them. The one with most is Media_Engage_Score, where approximately 10% of the total number of values is null. This quantity is not irrelevant, so preprocessing strategies like imputation will be needed. Finally, there are no constant columns.

Let's continue analyzing each type of variables. Starting with categorical variables, we must check which ones contain a high number of unique values. This is because in those cases an specific strategie should be followed in order to prevent overfitting and dimensional issues.

In [47]:
# Select categorical variables
categ_cols = df.select_dtypes("object").columns

# Cardinality 
cardinality = []

for c in categ_cols:
    cardinality.append({
        "Variable": c,
        "Cardinality": df[c].nunique()
    })

print("Cardinality of categorical values:\n ")
print(pd.DataFrame(cardinality).to_string(index=False))

Cardinality of categorical values:
 
      Variable  Cardinality
   Customer_ID        20240
        Gender            3
Signup_Quarter            4
       Country            8
          City           40


We see three different situations. First, *Gender* and *Signup_Quarter* exhibit low cardinality (3 and 4 respectively), so they are well suitable. In opposition, *Country* and *City* show high cardinality, especially the second one with 40 different values. Finally, obviously *Custiomer_ID* represents an identification, with one different value per costumer.

Talking about the target variable, *Canceled*, it is a binary variable indicating whether the customer has canceled the subscription (1) or remains active (0). Because of that, we are dealing with **Binary Classification Problem**. 

One of the first things we should study is if target variable is imbalanced, which means counting how much individuals we have from each class and compare them. For doing so, we begin transforming this variable into object type (it was int64).

In [48]:
df["Canceled"]=df["Canceled"].astype(object)

In [49]:
df["Canceled"].value_counts()

Canceled
0    10120
1    10120
Name: count, dtype: int64

We have exactly the same amount of instances for each class, so the target variable is perfectly balanced. As a consequence, metrics such as accuracy are reliable and sampling techniques to fix imbalance are not required.

### Univariated analysis